# 🏆 DATATHON 2026 — GenCore v6 | The Top-Kill Auxiliary Architecture
**Auxiliary Prophet Forecasting → XGBoost+LightGBM**
This notebook generates predicted features (orders, items, traffic) for 2023-2024 to feed the core GBDT model.

In [ ]:
import os, warnings, gc, glob
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

KAGGLE = os.path.exists('/kaggle/input')
if KAGGLE:
    matches = glob.glob('/kaggle/input/**/sales.csv', recursive=True)
    if not matches:
        raise FileNotFoundError('sales.csv not found')
    DATA_DIR = os.path.dirname(matches[0])
    OUT_DIR  = '/kaggle/working'
else:
    DATA_DIR = 'data/raw'
    OUT_DIR  = 'output'
print(f"ENV = {'Kaggle' if KAGGLE else 'Local'} | DATA = {DATA_DIR}")

In [ ]:
sales = pd.read_csv(f'{DATA_DIR}/sales.csv')
sales['Date'] = pd.to_datetime(sales['Date'])
# FILTER OUT OLD REGIME: Keep only 2020 onwards
sales = sales[sales['Date'].dt.year >= 2020]
sales = sales.sort_values('Date').reset_index(drop=True)

sub_tpl = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
sub_tpl['Date'] = pd.to_datetime(sub_tpl['Date'])
forecast_dates = sub_tpl['Date'].tolist()

print(f"Train: {sales.Date.min().date()} → {sales.Date.max().date()} ({len(sales)} rows)")
print(f"Fcast: {forecast_dates[0].date()} → {forecast_dates[-1].date()} ({len(forecast_dates)} rows)")

# ── Transaction & Traffic ──
wt = pd.read_csv(f'{DATA_DIR}/web_traffic.csv')
wt['date'] = pd.to_datetime(wt['date'])
wt_daily = wt.groupby('date', as_index=False)[['sessions', 'page_views']].mean().rename(columns={'date':'Date'})

orders = pd.read_csv(f'{DATA_DIR}/orders.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])
d_ord = orders.groupby('order_date').agg(daily_orders=('order_id','nunique')).reset_index().rename(columns={'order_date':'Date'})

oi = pd.read_csv(f'{DATA_DIR}/order_items.csv')
oi = oi.merge(orders[['order_id','order_date']], on='order_id', how='left')
d_items = oi.groupby('order_date').agg(daily_items=('product_id','count')).reset_index().rename(columns={'order_date':'Date'})

In [ ]:
print("\n═══ PHASE 1: Auxiliary Forecasting ═══")
from prophet import Prophet

# Vietnam Holidays
tet_dates = pd.DataFrame({
    'holiday': 'tet',
    'ds': pd.to_datetime(['2013-02-10','2014-01-31','2015-02-19','2016-02-08',
                          '2017-01-28','2018-02-16','2019-02-05','2020-01-25',
                          '2021-02-12','2022-02-01','2023-01-22','2024-02-10']),
    'lower_window': -3, 'upper_window': 7,
})
sale_events = pd.DataFrame({
    'holiday': 'mega_sale',
    'ds': pd.to_datetime([f'{y}-{m}-{d}' for y in range(2012,2025)
                          for m, d in [('04','30'),('05','01'),('09','02'),('11','11'),('12','12')]
                          if pd.Timestamp(f'{y}-{m}-{d}') <= pd.Timestamp('2024-07-01')]),
    'lower_window': -1, 'upper_window': 1,
})
holidays = pd.concat([tet_dates, sale_events], ignore_index=True)

aux_preds = pd.DataFrame({'Date': forecast_dates})
aux_features = ['daily_orders', 'daily_items', 'sessions', 'page_views']
aux_data = sales[['Date']].merge(d_ord, on='Date', how='left')\
                          .merge(d_items, on='Date', how='left')\
                          .merge(wt_daily, on='Date', how='left').fillna(0)

for feat in aux_features:
    print(f"Training Prophet for: {feat}...")
    df_prophet = aux_data[['Date', feat]].rename(columns={'Date':'ds', feat:'y'})
    m = Prophet(holidays=holidays, yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    m.fit(df_prophet)
    
    future = pd.DataFrame({'ds': forecast_dates})
    fc = m.predict(future)
    aux_preds[feat] = fc['yhat'].clip(lower=0).values

print("Auxiliary forecasting complete ✓")

In [ ]:
print("\n═══ PHASE 2: Feature Engineering ═══")

def build_features(df_sales, df_aux=None):
    df = df_sales.copy()
    if df_aux is not None:
        # Only merge columns that are not already in df (except Date)
        cols_to_use = ['Date'] + [c for c in df_aux.columns if c not in df.columns and c != 'Date']
        df = df.merge(df_aux[cols_to_use], on='Date', how='left')
    
    d = df['Date']
    df['year']         = d.dt.year
    df['month']        = d.dt.month
    df['day']          = d.dt.day
    df['dayofweek']    = d.dt.dayofweek
    df['dayofyear']    = d.dt.dayofyear
    df['weekofyear']   = d.dt.isocalendar().week.astype(int)
    df['quarter']      = d.dt.quarter
    df['is_weekend']   = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = d.dt.is_month_start.astype(int)
    df['is_month_end']   = d.dt.is_month_end.astype(int)
    df['is_tet']        = df['month'].isin([1,2]).astype(int)
    df['is_payday']     = df['day'].isin([1,15]).astype(int)
    
    for k in [1,2,3]:
        df[f'sin_y{k}'] = np.sin(2*np.pi*k*df['dayofyear']/365.25)
        df[f'cos_y{k}'] = np.cos(2*np.pi*k*df['dayofyear']/365.25)
    
    for t in ['Revenue', 'COGS']:
        s = df[t]
        for lag in [7, 14, 21, 28, 364]:
            df[f'{t}_lag{lag}'] = s.shift(lag)
        shifted = s.shift(1)
        for w in [7, 14, 28]:
            df[f'{t}_rmean{w}'] = shifted.rolling(w, min_periods=1).mean()
        df[f'{t}_ewm7']  = shifted.ewm(span=7, min_periods=1).mean()
        df[f'{t}_ewm28'] = shifted.ewm(span=28, min_periods=1).mean()
        
    return df

train = build_features(sales, aux_data)
lag_cols = [c for c in train.columns if '_lag' in c and '364' not in c]
train = train.dropna(subset=lag_cols).reset_index(drop=True)

feat_cols = sorted([c for c in train.columns if c not in ['Date', 'Revenue', 'COGS']])
X_train = train[feat_cols].fillna(0)
y_rev   = train['Revenue']
y_cogs  = train['COGS']

print(f"Training matrix: {X_train.shape[0]} rows × {len(feat_cols)} features")

In [ ]:
print("\n═══ PHASE 3: Core Model Training ═══")
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

def get_xgb():
    return XGBRegressor(n_estimators=1000, learning_rate=0.015, max_depth=6, 
                        subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1)
def get_lgb():
    return LGBMRegressor(n_estimators=1000, learning_rate=0.015, num_leaves=31, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbosity=-1)

xgb_rev = get_xgb().fit(X_train, y_rev)
lgb_rev = get_lgb().fit(X_train, y_rev)
xgb_cogs = get_xgb().fit(X_train, y_cogs)
lgb_cogs = get_lgb().fit(X_train, y_cogs)
print("Models trained successfully ✓")

imp = pd.Series(xgb_rev.feature_importances_, index=feat_cols).sort_values(ascending=False)
print("\nTop 10 features (XGBoost):")
print(imp.head(10).to_string())

In [ ]:
print(f"\n═══ PHASE 4: Recursive Forecast ({len(forecast_dates)} days) ═══")

hist_cols = ['Date', 'Revenue', 'COGS'] + aux_features
history = train[hist_cols].copy().sort_values('Date').tail(400).reset_index(drop=True)
aux_preds_indexed = aux_preds.set_index('Date')

predictions = []
for i, fdate in enumerate(forecast_dates):
    row = pd.DataFrame([{'Date': fdate, 'Revenue': np.nan, 'COGS': np.nan}])
    for c in aux_features:
        row[c] = aux_preds_indexed.at[fdate, c]
        
    combined = pd.concat([history, row], ignore_index=True)
    combined = build_features(combined, aux_preds)
    
    X_f = combined.iloc[-1:][feat_cols].fillna(0)
    
    rev_p = max(0, 0.5*xgb_rev.predict(X_f)[0] + 0.5*lgb_rev.predict(X_f)[0])
    cogs_p = max(0, 0.5*xgb_cogs.predict(X_f)[0] + 0.5*lgb_cogs.predict(X_f)[0])
    
    predictions.append({'Date': fdate, 'Revenue': rev_p, 'COGS': cogs_p})
    
    new_row = pd.DataFrame([{'Date': fdate, 'Revenue': rev_p, 'COGS': cogs_p}])
    for c in aux_features: new_row[c] = aux_preds_indexed.at[fdate, c]
    
    history = pd.concat([history, new_row], ignore_index=True).tail(400).reset_index(drop=True)
    if (i+1) % 100 == 0: print(f"  {i+1}/{len(forecast_dates)} done...")

pred_df = pd.DataFrame(predictions)
print("Forecast complete ✓")

In [ ]:
submission = sub_tpl[['Date']].copy()
submission = submission.merge(pred_df, on='Date', how='left')
submission['Revenue'] = submission['Revenue'].clip(lower=0).round(2)
submission['COGS']    = submission['COGS'].clip(lower=0).round(2)
submission['Date']    = submission['Date'].dt.strftime('%Y-%m-%d')

os.makedirs(OUT_DIR, exist_ok=True)
out_path = f'{OUT_DIR}/submission_v6_auxiliary.csv'
submission.to_csv(out_path, index=False, float_format='%.2f')

print(f"\n✅ Submission saved: {out_path}")
print(f"   Revenue: mean={submission.Revenue.mean():,.0f}")
print(submission.head(10))